In [45]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from joblib import dump
import json
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

param_grid = {
    "n_estimators": [100, 300, 500], 
    "criterion": ["gini", "entropy"],           
    "min_samples_split": [2, 5, 10],           
    "min_samples_leaf": [1, 2, 4],              
    "max_features": ["auto", "sqrt", "log2"],  
    "max_depth": [None, 10, 20, 30],             
    "bootstrap": [True, False]  
}

In [46]:
param_grid = {
    "n_estimators": [100], 
    "criterion": ["squared_error"],           
    "min_samples_split": [2],           
    "min_samples_leaf": [1],              
    "max_features": ["1.0"],  
    "max_depth": [10]
}

In [47]:
def metrics(predict_val, y_val, dataset, div):
    mae_value = mean_absolute_error(y_pred=predict_val, y_true=y_val)
    mse_value = mean_squared_error(y_pred=predict_val, y_true=y_val)
    rmse_value = np.sqrt(mse_value)
    r2_value = r2_score(y_pred=predict_val, y_true=y_val)

    df_metrics = pd.DataFrame(
        [[dataset, "Random_Forest_Regressor", div, mae_value, mse_value, rmse_value, r2_value]],
        columns=["dataset", "model", "sampling", "MAE", "MSE", "RMSE", "R2"]
    )

    return df_metrics

In [48]:
def split(df_data, seed):
    #Separa los datos
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    return train_data, val_data

In [49]:
def train(train, val, div, seed):
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()

    print(f"Train Regression Random Forest with seed {seed} and division {div}")
    results = []
    rf = RandomForestRegressor(random_state=seed)
    rf.fit(X_train, y_train)

    dump(rf, f"../../models/RandomForest_Regression_Grid_{seed}_{div}.joblib")

    y_pred_train = rf.predict(X_train)
    y_pred_val = rf.predict(X_val)

    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/RandomForest_Regression_Grid_{seed}_{div}_predictions.csv", index=False)
    
    train_metrics = metrics(y_pred_train, y_train, "Train", div)
    val_metrics = metrics(y_pred_val, y_val, "Validation", div)
    results.append(train_metrics)
    results.append(val_metrics)
    return pd.concat(results, ignore_index=True), rf

In [50]:
def grid_function(rf, train, val, seed, div): 
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()

    print(f"GridSearchCV for Regression Random Forest with seed {seed} and division {div}")
    grid = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_

    print(f"Best estimators: {best_model}")
    print(f"Best parameters found: {best_params}")
    best_params = json.dumps(best_params, indent=4)
    with open(f"../../models/RandomForest_Regression_Grid_{seed}_{div}_best_params.json", "w") as f:
        f.write(best_params)
    dump(best_model, f"../../models/RandomForest_Regression_Grid_{seed}_{div}_best.joblib")

    y_pred_train = best_model.predict(X_train)
    y_pred_val = best_model.predict(X_val)

    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/RandomForest_Regression_Grid_{seed}_{div}_best_predictions.csv", index=False)

    train_metrics = metrics(y_pred_train, y_train, "Train", div)
    val_metrics = metrics(y_pred_val, y_val, "Validation", div)
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    return results

In [51]:
def main_train(df_data, seed, grid_search=False):
    df_train, df_val = split(df_data, seed)
    metrics_orig, model_orig = train(df_train, df_val, "Original", seed)
    metrics_orig.to_csv(f"../../metrics/RandomForest_Regression_Grid_{seed}_metrics.csv", index=False)
    if grid_search:
        metrics_grid=grid_function(model_orig, df_train, df_val, seed, "Original"),
        metrics_grid = pd.concat(metrics_grid, ignore_index=True)
        metrics_grid.to_csv(f"../../metrics/RandomForest_Regression_Grid_{seed}_grid_metrics.csv", index=False)
    

In [52]:
repr_name="antiviral_homology_90_Q_prot5_embedding"
df_data = pd.read_csv(f"../../data/numerical_rep_reg/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

In [53]:
folder = "../../data/numerical_rep_reg/"
seed= 42

In [54]:
print(f"Processing {repr_name}")
main_train(df_data, seed, grid_search=True)
print(f"Finished processing {repr_name}")
print("=====================================")

Processing antiviral_homology_90_Q_prot5_embedding
Train Regression Random Forest with seed 42 and division Original
GridSearchCV for Regression Random Forest with seed 42 and division Original


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\hantr\anaconda3\envs\ML_Class\lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\hantr\anaconda3\envs\ML_Class\lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "c:\Users\hantr\anaconda3\envs\ML_Class\lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "c:\Users\hantr\anaconda3\envs\ML_Class\lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
